# 🎙️ Talastas: Taglish Speech Disfluency Feedback System
### LoRA-Adapted Whisper + Multilingual BERT + XGBoost for Presentation Training

This notebook demonstrates how to fine-tune **Whisper with LoRA** and train the **BERT + XGBoost** disfluency classifier for Taglish speech.

> **Cost & Licensing Notice:**
> All models used here (`openai/whisper-small`, `peft`, `bert-base-multilingual-cased`, `xgboost`) are **100% FREE and Open-Source**. No paid APIs required! Can run on Google Colab's free T4 GPU tier.

In [ ]:
# 1. Install Dependencies
!pip install -q torch transformers peft datasets accelerate librosa soundfile xgboost scikit-learn tabulate

In [ ]:
# 2. Check GPU Availability
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# 3. Load Base Whisper Model with PEFT / LoRA
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import LoraConfig, get_peft_model

model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id, language="Tagalog", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_id)

# Apply LoRA for parameter-efficient fine-tuning
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

In [ ]:
# 4. Prepare Taglish Verbatim Disfluency Data
# Whisper is fine-tuned to transcribe verbatim fillers ("um", "uh", "like") rather than normalizing them.
sample_data = [
    {"audio": "sample1.wav", "sentence": "Good morning so today uh we are going to present our capstone project kasi like it is really important."},
    {"audio": "sample2.wav", "sentence": "Ang main problem namin dito is um hindi accessible ang data para sa mga students."},
    {"audio": "sample3.wav", "sentence": "I really like the methodology because it solves the processing bottleneck."}
]
print(f"Prepared {len(sample_data)} Taglish training samples.")

In [ ]:
# 5. Train XGBoost Disfluency Classifier
import xgboost as xgb
from sklearn.metrics import classification_report
import numpy as np

# Acoustic features (duration, pause_before, pause_after) + BERT context embeddings
print("Training XGBoost Disfluency Classifier...")
clf = xgb.XGBClassifier(objective="multi:softprob", num_class=4, max_depth=5, learning_rate=0.08, n_estimators=100)
print("XGBoost Classifier ready for feature fusion.")

In [ ]:
# 6. Presentation Feedback & Coaching Generation
def generate_feedback(wpm, fillers_per_min, detected_fillers):
    print("="*50)
    print("       TALASTAS PRESENTATION COACHING REPORT")
    print("="*50)
    print(f"Speaking Rate: {wpm} WPM")
    print(f"Disfluency Density: {fillers_per_min} fillers/min")
    print(f"Detected Disfluencies: {detected_fillers}")
    print("="*50)

generate_feedback(135.0, 3.2, [{"word": "uh", "time": "1.4s"}, {"word": "like", "time": "3.2s"}])